In [1]:
from keras.models import load_model
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from skimage.transform import resize
import os

C:\ProgramData\Anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


# Cargar el modelo

In [2]:
model = load_model("leucemia_mnist.h5")

# Prediccion de una nueva imagen

In [3]:
images=[]
# AQUI ESPECIFICAMOS UNAS IMAGENES
dirname = os.path.join(os.getcwd(), 'datos/Val/validation_data')
filenames = [dirname+'/1.bmp']
print (filenames)

['C:\\Users\\Laura Valbuena\\Documents\\Proyecto\\M1000-IA--LLA-\\datos/Val/validation_data/1.bmp']


In [4]:
for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (257, 257),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

In [5]:
X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 257.

In [6]:
predicted_classes = model.predict(test_X)
estado=['Malignant','Benign']
print(predicted_classes)

1/1 [==============================] - 0s 288ms/step
[[0.8782082  0.12179184]]


In [7]:
for i, img_tagged in enumerate(predicted_classes):
    predict = estado[img_tagged.tolist().index(max(img_tagged))]
    print(filenames[i]+ " .... "+predict)

C:\Users\Laura Valbuena\Documents\Proyecto\M1000-IA--LLA-\datos/Val/validation_data/1.bmp .... Malignant


# Prediccion de varias imagenes de validación

Se crea el dataframe donde se guardaran las predicciones de cada imagen para luego comparar con el diagnostico real de la imagen 

In [8]:
df_predicted = pd.DataFrame()

# creamos las columnas
df_predicted['Filename'] = None
df_predicted['Prediccion'] = None

1. Recorremos el directorio que contiene 1867 imagenes, se realiza el reajuste del tamaño, se realiza la predicción y se guarda el resultado por cada imagen

2. El modelo tiene como respuesta ***Malignant** o **Benign**, sin embargo como el dataframe que tiene el diagnostico correcto, el dagnostico lo tiene en valores de 1 y 0 donde:

* 1 - Malignant

* 0 - Benign

Dentro de este recorrido hacemos un casteo para que convierta la salida del modelo en 1 o 0, para luego validar el resultado correcto, con el predicho  

In [9]:
dirname = os.path.join(os.getcwd(), 'datos')
imgpath = dirname + os.sep 

images = []
directories = []
dircount = []
prevRoot=''
cant=0

print("leyendo imagenes de ",imgpath)

for root, dirnames, filenames in os.walk(imgpath):
    head_tail  = os.path.split(root)
    if head_tail[1] == 'validation_data':
        for filename in filenames:
            if re.search("\.(jpg|jpeg|png|bmp|tif)$", filename):
                images = []
                cant=cant+1
                filepath = os.path.join(root, filename)
                image = plt.imread(filepath)
                image = plt.imread(filepath,0)
                image_resized = resize(image, (257, 257),anti_aliasing=True,clip=False,preserve_range=True)
                images.append(image_resized)
                X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
                #images.clear()
                test_X = X.astype('float32')
                test_X = test_X / 257.
                predicted_classes = model.predict(test_X)
                for i, img_tagged in enumerate(predicted_classes):
                    predict = estado[img_tagged.tolist().index(max(img_tagged))] 
                    if predict == 'Malignant':
                        predict = 1
                    else :
                        predict = 0
                    df_predicted = df_predicted.append({'Filename' : filename,'Prediccion' : predict}, ignore_index=True)
                    print("Validando: "+ filename+" -> Prediccion: "+str(predict))
                b = "Leyendo..."+ str(cant)
                print (b, end="\r")
                if prevRoot !=root:
                    print(root, cant)
                    prevRoot=root
                    directories.append(root)
                    dircount.append(cant)
                    cant=0
dircount.append(cant)

dircount = dircount[1:]
dircount[0]=dircount[0]+1
print('Directorios leidos:',len(directories))
print("Imagenes en cada directorio", dircount)
print('suma Total de imagenes en subdirs:',sum(dircount))

leyendo imagenes de  C:\Users\Laura Valbuena\Documents\Proyecto\M1000-IA--LLA-\datos\
1/1 [==============================] - 0s 59ms/step
Validando: 1.bmp -> Prediccion: 1
C:\Users\Laura Valbuena\Documents\Proyecto\M1000-IA--LLA-\datos\Val\validation_data 1


C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\142001448.py:35: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_predicted = df_predicted.append({'Filename' : filename,'Prediccion' : predict}, ignore_index=True)


1/1 [==============================] - 0s 61ms/step
Validando: 10.bmp -> Prediccion: 1
1/1 [==============================] - ETA: 0s

C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\142001448.py:35: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_predicted = df_predicted.append({'Filename' : filename,'Prediccion' : predict}, ignore_index=True)


1/1 [==============================] - 0s 65ms/step
Validando: 100.bmp -> Prediccion: 1
1/1 [==============================] - ETA: 0s

...

C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\142001448.py:35: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_predicted = df_predicted.append({'Filename' : filename,'Prediccion' : predict}, ignore_index=True)


1/1 [==============================] - 0s 73ms/step
Validando: 998.bmp -> Prediccion: 1


C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\142001448.py:35: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_predicted = df_predicted.append({'Filename' : filename,'Prediccion' : predict}, ignore_index=True)


1/1 [==============================] - 0s 84ms/step
Validando: 999.bmp -> Prediccion: 1
Directorios leidos: 1
Imagenes en cada directorio [1867]
suma Total de imagenes en subdirs: 1867


C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\142001448.py:35: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_predicted = df_predicted.append({'Filename' : filename,'Prediccion' : predict}, ignore_index=True)


In [10]:
df_predicted

,Filename,Prediccion
0,1.bmp,1
1,10.bmp,1
2,100.bmp,1
3,1000.bmp,1
4,1001.bmp,1
...,...,...
1862,995.bmp,1
1863,996.bmp,1
1864,997.bmp,1
1865,998.bmp,1


# Validar Predicciones
Se carga el dataframe que contiene el nombre de la imagen y el pronostico correcto para la imagen en la columna labels donde:

* 1 - Malignant

* 0 - Benign

In [11]:
dirname = os.path.join(os.getcwd(), 'datos/Val/validation_data/')
datos = pd.read_csv(dirname +'labels_images.csv',sep=',')
datos

,Patient_ID,new_names,labels
0,UID_57_29_1_all.bmp,1.bmp,1
1,UID_57_22_2_all.bmp,2.bmp,1
2,UID_57_31_3_all.bmp,3.bmp,1
3,UID_H49_35_1_hem.bmp,4.bmp,0
4,UID_58_6_13_all.bmp,5.bmp,1
...,...,...,...
1862,UID_54_33_1_all.bmp,1863.bmp,1
1863,UID_55_24_1_all.bmp,1864.bmp,1
1864,UID_H32_20_1_hem.bmp,1865.bmp,0
1865,UID_54_30_2_all.bmp,1866.bmp,1


In [12]:
#Se realiza el Join del dataframe con los resultados correctos y el dataframe con la predicción
df_join = datos.set_index('new_names').join(df_predicted.set_index('Filename')).reset_index()
df_join['Valor'] = None
df_join

,new_names,Patient_ID,labels,Prediccion,Valor
0,1.bmp,UID_57_29_1_all.bmp,1,1,None
1,2.bmp,UID_57_22_2_all.bmp,1,1,None
2,3.bmp,UID_57_31_3_all.bmp,1,1,None
3,4.bmp,UID_H49_35_1_hem.bmp,0,1,None
4,5.bmp,UID_58_6_13_all.bmp,1,1,None
...,...,...,...,...,...
1862,1863.bmp,UID_54_33_1_all.bmp,1,1,None
1863,1864.bmp,UID_55_24_1_all.bmp,1,1,None
1864,1865.bmp,UID_H32_20_1_hem.bmp,0,1,None
1865,1866.bmp,UID_54_30_2_all.bmp,1,1,None


In [13]:
for i in range(len(df_join)):
    if df_join['labels'][i] == df_join['Prediccion'][i]:
        df_join['Valor'][i] = 'Correct'
    else:
        df_join['Valor'][i] = 'Incorrect'

C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\3056822646.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_join['Valor'][i] = 'Correct'
C:\Users\Laura Valbuena\AppData\Local\Temp\ipykernel_25856\3056822646.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_join['Valor'][i] = 'Incorrect'


In [14]:
df_join

,new_names,Patient_ID,labels,Prediccion,Valor
0,1.bmp,UID_57_29_1_all.bmp,1,1,Correct
1,2.bmp,UID_57_22_2_all.bmp,1,1,Correct
2,3.bmp,UID_57_31_3_all.bmp,1,1,Correct
3,4.bmp,UID_H49_35_1_hem.bmp,0,1,Incorrect
4,5.bmp,UID_58_6_13_all.bmp,1,1,Correct
...,...,...,...,...,...
1862,1863.bmp,UID_54_33_1_all.bmp,1,1,Correct
1863,1864.bmp,UID_55_24_1_all.bmp,1,1,Correct
1864,1865.bmp,UID_H32_20_1_hem.bmp,0,1,Incorrect
1865,1866.bmp,UID_54_30_2_all.bmp,1,1,Correct


In [15]:
df_join.to_csv('leucemia_Pronostico.csv', index=False, sep=';')

In [16]:
df_join.groupby("Valor", sort=False).describe()

labels                                             
            count      mean       std  min  25%  50%  75%  max
Valor                                                         
Correct    1314.0  0.899543  0.300722  0.0  1.0  1.0  1.0  1.0
Incorrect   553.0  0.066908  0.250088  0.0  0.0  0.0  0.0  1.0

In [17]:
df_join.groupby('Valor').size().reset_index().rename(columns={0:'count'})

,Valor,count
0,Correct,1314
1,Incorrect,553


De el total de las imagenes (1867), el 89.9% tuvo una predicción correcta correspondiete a 1314 imagenes